# Practical 03: In-Memory Optimization via Dtype Downcasting & Deep Profiling

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to demonstrate in-memory optimization of large datasets. Profile baseline memory usage using memory_usage(deep=True), downcast integer columns (int64 -> int32/int16), downcast float columns (float64 -> float32), and convert repetitive string columns to category type. Report the total percentage reduction in RAM footprint.

In [1]:
import numpy as np
import pandas as pd

rg = np.random.default_rng(99)
n = 700_000
frame = pd.DataFrame({
    "transaction_id": np.arange(1, n + 1, dtype="int64"),
    "customer_id": rg.integers(1, 55000, n),
    "store_id": rg.integers(1, 240, n),
    "quantity": rg.integers(1, 18, n),
    "amount": np.round(rg.gamma(2.2, 65.0, n), 2),
    "category": rg.choice(["Electronics", "Clothing", "Groceries", "Furniture", "Toys", "Beauty"], n).astype(object),
    "payment_method": rg.choice(["Credit Card", "Debit Card", "UPI", "Cash", "Net Banking"], n).astype(object),
    "city": rg.choice(["Kolkata", "Patna", "Ranchi", "Delhi", "Lucknow", "Surat", "Ahmedabad"], n).astype(object),
})
print(frame.head())

def mb(frame):
    """Total deep memory of a DataFrame in MB."""
    return frame.memory_usage(deep=True).sum() / 1024 ** 2

before_mb = mb(frame)
print(frame.dtypes, "\n")
print("Memory per column (MB):")
print((frame.memory_usage(deep=True) / 1024 ** 2).round(3))
print(f"\nBASELINE TOTAL: {before_mb:.2f} MB")

lean = frame.copy()
for col in lean.select_dtypes(include="integer").columns:
    lean[col] = pd.to_numeric(lean[col], downcast="integer")

after_int = mb(lean)
print(lean.select_dtypes(include="integer").dtypes.to_dict())
print(f"After integer downcast: {after_int:.2f} MB")

for col in lean.select_dtypes(include="float").columns:
    lean[col] = pd.to_numeric(lean[col], downcast="float")

after_float = mb(lean)
print(lean.select_dtypes(include="float").dtypes.to_dict())
print(f"After float downcast: {after_float:.2f} MB")

for col in lean.select_dtypes(include=["object", "string"]).columns:
    if lean[col].nunique() / len(lean) < 0.25:
        lean[col] = lean[col].astype("category")

after_cat = mb(lean)
print(lean.dtypes)
print(f"\nAfter category conversion: {after_cat:.2f} MB")

reduction = (before_mb - after_cat) / before_mb * 100
print("=" * 45)
print(f"Baseline  : {before_mb:8.2f} MB")
print(f"Optimised : {after_cat:8.2f} MB")
print(f"REDUCTION : {reduction:8.2f} %")
print("=" * 45)

,transaction_id,customer_id,store_id,quantity,amount,category,payment_method,city
0,1,52699,7,8,223.33,Groceries,Credit Card,Ranchi
1,2,27832,40,16,376.84,Furniture,Net Banking,Surat
2,3,41671,127,9,257.55,Groceries,Credit Card,Patna
3,4,31080,148,5,52.92,Electronics,Net Banking,Patna
4,5,9780,210,17,242.30,Beauty,Cash,Surat


transaction_id      int64
customer_id         int64
store_id            int64
quantity            int64
amount            float64
category           object
payment_method     object
city               object
dtype: object 

Memory per column (MB):
Index              0.000
transaction_id     5.341
customer_id        5.341
store_id           5.341
quantity           5.341
amount             5.341
category          37.938
payment_method    37.918
city              36.907
dtype: float64

BASELINE TOTAL: 139.47 MB
{'transaction_id': dtype('int32'), 'customer_id': dtype('int32'), 'store_id': dtype('int16'), 'quantity': dtype('int8')}
After integer downcast: 125.45 MB
{'amount': dtype('float32')}
After float downcast: 122.78 MB
transaction_id       int32
customer_id          int32
store_id             int16
quantity              int8
amount             float32
category          category
payment_method    category
city              category
dtype: object

After category conversion: 12.02 MB
Ba